In [ ]:
import os
import json
import shutil
import pandas as pd

from pathlib import Path
from collections import Counter

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict

# MILK10k Paths

In [ ]:
MILK10K_DOWNLOAD_PATH = Path("/home/sulcm/datasets/milk10k/milk10k_downloaded")
MILK10K_BUILDER = Path("/home/sulcm/datasets/milk10k/milk10k_builder")
MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/milk10k")
MILK10K_TEST_BUILDER = Path("/home/sulcm/datasets/milk10k/milk10k_test_builder")
MILK10K_TEST_DATASET = Path("/home/sulcm/datasets/milk10k/TestMILK10k")
SPLIT_MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/SpilledMILK10k")
OLD_SPLIT_MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/OLD_SpilledMILK10k")
AUGMENTED_MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/AugmentedMILK10k")

## Create dataset structure

In [ ]:
ds_gt = pd.read_csv(MILK10K_DOWNLOAD_PATH / "training_gt.csv").set_index("lesion_id", append=True)
labels = [l.lower() for l in ds_gt.columns.to_list()]
ds_gt_classes: pd.Series = ds_gt.dot(ds_gt.columns).apply(lambda x: x.lower())

In [ ]:
ben_cls = ["nv", "bkl", "df", "inf", "vasc", "ben_oth"]
mal_cls = ["bcc", "sccka", "mel", "akiec", "mal_oth"]

In [ ]:
counts_per_cls = ds_gt_classes.value_counts() * 2

In [ ]:
total_samples = counts_per_cls.sum()
total_samples

In [ ]:
sum(counts_per_cls[c] for c in ben_cls)

In [ ]:
sum(counts_per_cls[c] for c in mal_cls)

In [ ]:
ds_training_input = pd.read_csv(MILK10K_DOWNLOAD_PATH / "milk10k" / "metadata.csv")
ds_training_input[["file_name", "label"]] = ds_training_input.apply(
    lambda row: [
        row["isic_id"] + ".jpg",
        ds_gt_classes.xs(row["lesion_id"], level=1).iloc[0]
    ],
    axis=1, result_type="expand"
)
ds_training_input.drop(columns=["attribution", "copyright_license"], inplace=True)

In [ ]:
dset_grouped_by_label = ds_training_input.groupby("label")

In [ ]:
dset_label_veiws = {}
for label, samples in dset_grouped_by_label.groups.items():
    dset_label_veiws[label] = ds_training_input.iloc[samples].groupby("lesion_id")

In [ ]:
dset_label_veiws

In [ ]:
dset_label_veiws["df"].groups

In [ ]:
ds_training_input.iloc[[2160, 6850]]

In [ ]:
ds_info = {
    "labels": labels
}

In [ ]:
# if not MILK10K_BUILDER.exists():
#     shutil.copytree(MILK10K_DOWNLOAD_PATH / "milk10k" / "images", MILK10K_BUILDER / "train")
#     ds_training_input.to_csv(MILK10K_BUILDER / "train" / "metadata.csv", index=False)
#     with open(MILK10K_BUILDER / "dataset_info.json", "w") as f:
#         json.dump(ds_info, f, indent=2, ensure_ascii=False)

## Load/Build datatset

In [ ]:
dataset = load_dataset("imagefolder", data_dir=MILK10K_BUILDER)
dataset

In [ ]:
dataset = dataset.cast_column("label", ClassLabel(names=labels))

In [ ]:
# dataset.save_to_disk(MILK10K_DATASET)

## Use build MILK10k dataset

In [ ]:
lds = load_from_disk(
    dataset_path=MILK10K_DATASET
)
lds

In [ ]:
# lds.cleanup_cache_files()

In [ ]:
lds["train"].features

In [ ]:
milk_shapes = [l_im.size for l_im in lds["train"]["image"]]

In [ ]:
Counter(milk_shapes)

## Split MILK10k dataset

In [ ]:
milk10k_split = lds["train"].train_test_split(
    # test_size=0.1,
    test_size=0.05,
    shuffle=True,
    stratify_by_column="label",
    seed=42
)
milk10k_split = DatasetDict({
    "train": milk10k_split["train"],
    "validation": milk10k_split["test"]
})
milk10k_split

In [ ]:
# milk10k_split.save_to_disk(SPLIT_MILK10K_DATASET)

In [ ]:
split_lds = load_from_disk(
    # dataset_path=OLD_SPLIT_MILK10K_DATASET
    dataset_path=SPLIT_MILK10K_DATASET
)
split_lds

In [ ]:
# split_lds.cleanup_cache_files()

# MILK10k TEST split

In [ ]:
milk10k_test_split_path = MILK10K_TEST_BUILDER / "test"
if not milk10k_test_split_path.exists():
    os.mkdir(milk10k_test_split_path)

In [ ]:
total_files = 0
for root, dirs, _ in os.walk(MILK10K_DOWNLOAD_PATH / "MILK10k_Test_Input"):
    for d in dirs:
        dir_path = os.path.join(root, d)
        files = os.listdir(dir_path)
        for file in files:
            if not file.endswith(".jpg"):
                continue
            total_files += 1
            shutil.copy(
                os.path.join(dir_path, file),
                milk10k_test_split_path
            )

total_files

In [ ]:
ds_test_input = pd.read_csv(MILK10K_DOWNLOAD_PATH / "MILK10k_Test_Metadata.csv")
ds_test_input["file_name"] = ds_test_input.apply(
    lambda row: row["isic_id"] + ".jpg", axis=1
)
ds_test_input = ds_test_input.filter(["lesion_id", "image_type", "isic_id", "age_approx", "sex", "skin_tone_class", "site", "file_name"])
ds_test_input.to_csv(milk10k_test_split_path / "metadata.csv", index=False)

In [ ]:
# test_dataset = load_dataset("imagefolder", data_dir=MILK10K_TEST_BUILDER)
# test_dataset

In [ ]:
# test_dataset.save_to_disk(MILK10K_TEST_DATASET)

In [ ]:
test_lds = load_from_disk(
    dataset_path=MILK10K_TEST_DATASET
)
test_lds

In [ ]:
# test_lds.cleanup_cache_files()

# Augmented MILK10k Dataset

In [ ]:
aug_lds = load_from_disk(
    dataset_path=AUGMENTED_MILK10K_DATASET
)
aug_lds

In [ ]:
# aug_lds.cleanup_cache_files()

In [ ]:
Counter(aug_lds["train"]["label"])